# Sprint 1 · Baseline mínimo (regla trivial) + logs

**Tesis:** Diagnóstico de causalidad de fallas en una red telco usando grafos · Región **Pasco**, Perú.

Objetivo (tarjeta Trello, fase **Modelado**): establecer un **punto de comparación trivial** que proponga una
causa raíz **Top-1 por caso** y deje **resultados y logs reproducibles**. Es el piso contra el cual se medirá
el motor híbrido (Hawkes topológico / GNN) en los próximos sprints. Reutiliza `src/baseline.py` (Opción B).

## 1. Definición de la regla trivial de referencia

- **Caso:** una *avalancha* = un equipo (`address`) con **≥ 3 alarmas** en una misma ventana.
- **Reglas** (triviales, no aprenden):
  - **`temprana`** → la alarma que **ocurrió primero** es la causa (la causa precede al efecto). *Regla por defecto.*
  - **`severa`** → la alarma **más severa** (menor `severity`).
  - **`frecuente`** → el **tipo de alarma más frecuente** del caso.
- **Salida:** Top-1 por caso (con ranking Top-k).
- **Evaluación provisional:** sin etiquetas del NOC aún, se usa un **proxy de dominio** — cuando co-ocurren
  `RADIO_RSL_LOW` y `MW_LOF`, la causa física es `RADIO_RSL_LOW`. *No es verdad de referencia validada*, es un
  control de cordura. La verdad de referencia real se construirá con el NOC (Sprint siguiente).

> ⚠️ **Actualización.** Tras separar los **sectoriales** (RTN510_AP) en la ingesta, este proxy ya **no tiene casos**:
> la pareja `RADIO_RSL_LOW` + `MW_LOF` solo co-ocurría en los radios hacia clientes. Por eso la exactitud del
> baseline queda pendiente de un **conjunto de referencia etiquetado** (ver §7).

## 2. Carga del consolidado

In [ ]:
import os, sys, json
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

proc = ROOT / "data" / "processed" / "dataset_consolidado.parquet"
if proc.exists():
    df = pd.read_parquet(proc)
else:
    from src.ingesta import construir_dataset, parsear_moi, marcar_particion_temporal
    df = marcar_particion_temporal(parsear_moi(construir_dataset(str(ROOT / "data" / "raw" / "Datos"))))
for col in ["firstOccurUtc", "arriveUtc", "occurUtc", "latestOccurUtc"]:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors="coerce")
print("Dataset:", df.shape)

> **Resumen.** Carga el mismo consolidado del EDA (ya filtrado) y normaliza las fechas. Es el insumo sobre el que el baseline construye los casos y propone la causa raíz.

## 3. Construcción de casos (avalanchas)

In [ ]:
from src.baseline import construir_casos, baseline_top1, evaluar_proxy, ejecutar

casos = construir_casos(df, umbral=3)
print(f"Casos (equipo x ventana con 3+ alarmas): {len(casos):,}")
print(f"Tamaño medio: {casos['n_alarmas'].mean():.1f} | máx: {casos['n_alarmas'].max()}")
casos.sort_values("n_alarmas", ascending=False).head(5)

> **Resumen.** Define los **casos a diagnosticar**: cada avalancha (equipo con ≥3 alarmas en una ventana). Imprime cuántos casos hay (**4 192**) y su tamaño medio/máximo. Cada caso recibirá una causa raíz Top-1.

## 4. Baseline Top-1 (regla por defecto: `temprana`)

In [ ]:
top1 = baseline_top1(df, regla="temprana", umbral=3, topk=3)
print("Top-1 por caso:", top1.shape)
display(top1[["ventana_id", "address", "n_alarmas", "top1_alarmName", "topk_alarmas"]].head(10))

# distribución de la causa raíz propuesta
print("\nCausa raíz Top-1 más propuesta (regla temprana):")
display(top1["top1_alarmName"].value_counts().head(10).to_frame("n_casos"))

> **Qué muestran las tablas.** *(Arriba)* la **causa raíz Top-1 propuesta por caso** con la regla `temprana` (columna `top1_alarmName`) y su ranking Top-k. *(Abajo)* qué tipos de alarma propone más como causa. Es la salida central del baseline: una hipótesis de causa por avalancha.

## 5. Medición del Top-1 inicial (proxy de dominio)

Comparamos las 3 reglas contra el proxy `RADIO_RSL_LOW → MW_LOF`. Con el dataset sin sectoriales el proxy no
tiene casos, así que la celda muestra `casos_proxy = 0` y la exactitud queda vacía (`None`).

In [ ]:
filas = []
for r in ["temprana", "severa", "frecuente"]:
    t = baseline_top1(df, regla=r, umbral=3)
    m = evaluar_proxy(t, df)
    filas.append({"regla": r, "n_casos": len(t), **m})
comp = pd.DataFrame(filas)
display(comp)

> **Qué muestra la tabla.** Para cada regla: cuántos casos recibieron un Top-1 (`n_casos` = **4 192**), cuántos casos tienen la pareja del proxy (`casos_proxy`) y la exactitud resultante. Con el dataset limpio `casos_proxy = 0` y la exactitud es `None`: el proxy dejó de servir como referencia.

## 6. Registrar resultados y logs

`ejecutar()` fija la semilla, deja logs y guarda en `results/baseline/`: un CSV de Top-1 por caso por regla y un
`baseline_metrics.json` con las métricas. Reproducible con un solo llamado.

In [ ]:
metrics = ejecutar(df, out_dir=str(ROOT / "results" / "baseline"), umbral=3)
print(json.dumps(metrics, indent=2, ensure_ascii=False))
print("\nArchivos en results/baseline/:", [p.name for p in (ROOT / "results" / "baseline").glob("baseline_*")])

> **Resumen.** Deja la **evidencia reproducible** del baseline: el JSON de métricas y los CSV de Top-1 por caso en `results/baseline/`, con semilla fija y logs. Esto cumple el DoD de la tarjeta (resultados y logs reproducibles).

## 7. Lectura del baseline (resultados reales, dataset sin sectoriales)

| Regla | Casos con Top-1 | Casos del proxy | Top-1 (proxy) |
|---|---|---|---|
| `temprana` | 4 192 | 0 | — |
| `severa` | 4 192 | 0 | — |
| `frecuente` | 4 192 | 0 | — |

- **El baseline funciona:** produce una causa raíz Top-1 (y su ranking Top-k) para cada uno de los **4 192 casos**
  y deja resultados y logs reproducibles en `results/baseline/`. Esa parte del DoD se cumple.
- **La medición de exactitud quedó sin referencia.** El proxy `RADIO_RSL_LOW → MW_LOF` tiene **0 casos** una vez
  separados los sectoriales: esa pareja solo aparecía en los radios hacia clientes. Por lo tanto, la cifra anterior
  (6–9 %) **se descarta**: medía equipos que no son parte del problema de la red.
- **La pareja dominante ahora (BGP) no sirve como proxy:** `BGP Status Changed` y `Bgp Peer Backward Transition`
  reportan el mismo evento, así que no definen cuál es la causa y cuál el efecto.
- **Consecuencia metodológica:** la exactitud Top-1/Top-k debe medirse contra un **conjunto de referencia**. Propuesta:
  1. **Etiqueta "plata"** a partir de los manuales de alarmas Huawei (qué alarma provoca cuáles), aplicada a todos los casos.
  2. **Etiqueta "oro"** del NOC sobre una muestra (p. ej. las avalanchas más grandes + una muestra aleatoria por red),
     que además sirve para validar la etiqueta plata.
  Esto se formaliza en la tarjeta *Protocolo de validación* del Sprint 2, y con él se medirán el baseline y el motor.
- **Señal de datos que se mantiene:** la regla `temprana` ordena por `firstOccurUtc` (primera ocurrencia histórica;
  hay alarmas crónicas). Para el orden temporal se usará `arriveUtc` y se tratarán las alarmas crónicas antes de Hawkes.
